# 🚀 Ultra-Resolution HPC Lid-Driven Cavity CFD Suite
### **Scaling to $1.05$ Million Nodes ($N = 1025$) & $Re = 10,000 \longrightarrow 100,000$ with Pure Zero-Dissipation ($\nu_{\text{num}} = 0$) Central Differencing**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KartikeyaGangwar/lid-driven-cavity-cfd/blob/main/cavity_hpc_colab.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/KartikeyaGangwar/lid-driven-cavity-cfd)
[![Zenodo Permanent DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18312938.svg)](https://doi.org/10.5281/zenodo.18312938)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**Author:** Kartikey Singh  
**Affiliation:** Independent Computational Engineering Researcher  
**Permanent Archive DOI:** [10.5281/zenodo.18312938](https://doi.org/10.5281/zenodo.18312938)

---

### **Executive Summary**
This interactive high-performance notebook executes reference-grade finite-difference simulations of the 2D incompressible lid-driven cavity problem at extreme resolutions ($N = 1025 \times 1025 = 1,050,625$ collocation nodes) across the high and extreme Reynolds number frontier ($Re = 10,000$ to $Re = 100,000$).

Key Computational Capabilities:
1. **Zero Numerical Dissipation ($\nu_{\text{num}} = 0$):** Pure 2nd-order central differencing preserves delicate shear layers and micro-eddies without artificial upwind damping.
2. **Exact Spectral Poisson Solver:** $\mathcal{O}(N^2 \log N)$ 2D Type-I Discrete Sine Transform (DST-I) achieves machine-precision streamfunction inversion in sub-second times on $1.05$ million nodes.
3. **Warm-Start Bicubic Continuation:** Resamples converged $N = 513$ flow fields directly to $N = 1025$, reducing steady relaxation runtime by $> 85\%$.
4. **Benchmark Auditing:** Direct automated verification against Dr. Ercan Erturk's Table 3 benchmark (*acenumerics.com*) at $y = 0.95$.
5. **Physical Time-Accurate Marching:** Captures primary Hopf bifurcation ($St = 0.6661$), 2-torus quasi-periodic attractors, and compiles synchronized 60-frame animated GIFs inline.


## 1. Hardware Inspection & Dependencies Setup
Check available Colab CPU/GPU resources and verify required scientific libraries (`numpy`, `scipy`, `matplotlib`, `pillow`).


In [ ]:
# Check Hardware Resources
import os
import sys
import multiprocessing
import platform

print("=" * 65)
print("COMPUTATIONAL PLATFORM TELEMETRY")
print("=" * 65)
print(f"Operating System:      {platform.platform()}")
print(f"Python Version:        {platform.python_version()}")
print(f"CPU Core Count:        {multiprocessing.cpu_count()} logical cores")

# Check GPU availability (if GPU runtime enabled in Colab)
try:
    import torch
    if torch.cuda.is_available():
        print(f"NVIDIA GPU Detected:   {torch.cuda.get_device_name(0)}")
        print(f"VRAM Total:            {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        print("NVIDIA GPU Detected:   None (Running on High-Performance CPU)")
except ImportError:
    print("PyTorch check skipped.")
print("=" * 65)

# Verify scientific computing stack (Colab comes pre-configured with optimized packages)
try:
    import numpy as np
    import scipy
    import matplotlib
    import PIL
    print(f"[OK] Scientific stack verified: NumPy {np.__version__} | SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")
except ImportError:
    !pip install --quiet "numpy<2.3,>=1.24" scipy matplotlib pillow


## 2. Clone / Sync the CFD Codebase
Clones the official GitHub repository `lid-driven-cavity-cfd` or pulls the latest updates if already present. Optionally mounts Google Drive to permanently persist heavy flow field data.


In [ ]:
# Mount Google Drive (Optional - uncomment to persist data across Colab restarts)
# from google.colab import drive
# drive.mount('/content/drive')

import os

repo_name = "lid-driven-cavity-cfd"
repo_url = "https://github.com/KartikeyaGangwar/lid-driven-cavity-cfd.git"

if not os.path.exists(repo_name):
    print(f"Cloning repository from {repo_url} ...")
    !git clone {repo_url}
    %cd {repo_name}
else:
    print("Repository directory already exists. Resetting and pulling latest commits...")
    %cd {repo_name}
    !git fetch origin
    !git reset --hard origin/main

print(f"Active working directory: {os.getcwd()}")


## 3. Boundary Layer Scaling & Resolution Physics ($\delta \sim 1/\sqrt{Re}$)

In 2D lid-driven cavity flows, the viscous boundary layer thickness adjacent to the walls scales as:
$$\delta \approx \frac{L}{\sqrt{Re}}$$

To properly resolve high-Re boundary layers without numerical oscillations or artificial numerical diffusion, the number of grid points spanning $\delta$ must satisfy:
$$N_{\text{bl}} = \frac{\delta}{h} = \frac{N - 1}{\sqrt{Re}} \ge 8 \sim 10$$

### **Discretization Comparison:**
- **On $N = 513$ ($h = 1/512$):** At $Re = 10,000$, $\sqrt{Re} = 100 \implies N_{\text{bl}} = 5.12$ points.
- **On $N = 1025$ ($h = 1/1024 \approx 9.77 \times 10^{-4}$):** At $Re = 10,000$, $N_{\text{bl}} = \mathbf{10.24}$ points!
- **Cell Peclet Number Invariance:**
  $$\text{At } Re = 50,000 \text{ on } N = 513: \quad Pe_h = \frac{50,000}{512} = \mathbf{97.66}$$
  $$\text{At } Re = 100,000 \text{ on } N = 1025: \quad Pe_h = \frac{100,000}{1024} = \mathbf{97.65}$$
Because the spatial cell Peclet number is mathematically identical, our central differencing scheme maintains the exact same numerical stability at $Re = 100,000$ on $N = 1025$ as demonstrated at $Re = 50,000$ on $N = 513$!


## 4. High-Order Bicubic Spline Prolongation Engine
Instead of iterating thousands of relaxation steps from quiescent initial conditions, we interpolate our converged $N = 513$ flow fields directly onto the $1.05$ Million node ($N = 1025$) mesh with exact physical wall boundary enforcement.


In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator

def prolong_coarse_to_ultra(source_npz, N_target=1025, L=1.0):
    '''
    Interpolates coarse flow field to ultra-fine N_target x N_target mesh.
    Enforces exact Dirichlet and no-slip wall closures.
    '''
    print(f"[PROLONG] Loading source file: {source_npz}")
    src = np.load(source_npz)
    x_src, y_src = src['x'], src['y']
    N_src = len(x_src)
    print(f"[PROLONG] Resampling from {N_src}x{N_src} -> {N_target}x{N_target} ({N_target**2:,} collocation nodes)...")

    x_tgt = np.linspace(0.0, L, N_target)
    y_tgt = np.linspace(0.0, L, N_target)
    X_tgt, Y_tgt = np.meshgrid(x_tgt, y_tgt, indexing='xy')
    query_pts = np.stack([Y_tgt.ravel(), X_tgt.ravel()], axis=-1)

    prolonged = {}
    for var in ['psi', 'omega', 'u', 'v']:
        if var in src:
            interp = RegularGridInterpolator((y_src, x_src), src[var], method='cubic',
                                             bounds_error=False, fill_value=None)
            prolonged[var] = interp(query_pts).reshape((N_target, N_target))

    # Exact boundary condition enforcement
    prolonged['psi'][0, :] = 0.0
    prolonged['psi'][-1, :] = 0.0
    prolonged['psi'][:, 0] = 0.0
    prolonged['psi'][:, -1] = 0.0

    prolonged['u'][0, :] = 0.0
    prolonged['u'][-1, :] = 1.0  # Constant moving lid
    prolonged['u'][:, 0] = 0.0
    prolonged['u'][:, -1] = 0.0

    prolonged['v'][0, :] = 0.0
    prolonged['v'][-1, :] = 0.0
    prolonged['v'][:, 0] = 0.0
    prolonged['v'][:, -1] = 0.0

    print("[PROLONG] Prolongation completed with exact wall boundary enforcement.")
    return prolonged, x_tgt, y_tgt


## 5. Execute Ultra-Resolution Steady Continuation ($N = 1025$)
Run steady relaxation on $1.05$ Million nodes for any chosen Reynolds number ($Re = 10,000$ to $Re = 100,000$).


In [ ]:
import os
import numpy as np
from run_ultra_resolution_1025 import solve_ultra_steady
from lid_driven_cavity_fdm import LidDrivenCavitySolver

# Configure target Reynolds number and source continuation checkpoint
TARGET_RE = 100000         # 1 Lakh Reynolds Number (Extreme Frontier!)
TARGET_N = 1025           # 1025 x 1025 (1,050,625 collocation nodes)

target_file = f"data/flow_fields_Re{TARGET_RE}_N{TARGET_N}.npz"

if os.path.exists(target_file):
    print(f"[CACHE HIT] Pre-converged solution found: {target_file}")
    print(f"[CACHE HIT] Loading {target_file} directly in 1 second... (No need to re-run steady solve!)")
    out_file = target_file
    solver = LidDrivenCavitySolver(N=TARGET_N, Re=TARGET_RE, poisson_solver='dst', wall_beta=0.50)
    data = np.load(target_file)
    solver.psi = data['psi'].copy()
    solver.omega = data['omega'].copy()
    solver.u = data['u'].copy()
    solver.v = data['v'].copy()
    solver.u_c = solver.u[1:-1, 1:-1].copy()
    solver.v_c = solver.v[1:-1, 1:-1].copy()
    print("[READY] Steady flow fields loaded successfully into solver!")
else:
    # Smart Continuation Ladder: Warm-start from Re=50,000 (N=513) converged flow field
    source_checkpoint = "data/flow_fields_Re50000_N513.npz"
    if not os.path.exists(source_checkpoint):
        source_checkpoint = f"data/flow_fields_Re{TARGET_RE}_N513.npz"
    if not os.path.exists(source_checkpoint):
        source_checkpoint = "data/flow_fields_Re10000_N513.npz"

    print(f"[HPC CONFIG] Target: Re = {TARGET_RE:,} on Mesh {TARGET_N}x{TARGET_N} ({TARGET_N**2:,} nodes)")
    print(f"[HPC CONFIG] Warm-start checkpoint: {source_checkpoint}")

    solver, out_file = solve_ultra_steady(
        Re=TARGET_RE,
        N=TARGET_N,
        source_file=source_checkpoint,
        max_iter=10000,
        tol=1e-6,
        check_interval=200
    )


## 6. High-Contrast Streamlines, Vorticity & Centerline Profiles at 1.05 Million Nodes
Generates a comprehensive 3-panel visualization:
1. **Clean Streamlines:** High-density continuous contours resolving primary, secondary (BR1, BL1, TL1), and quaternary eddies with **zero blank space and no marker clutter**.
2. **Symmetric Logarithmic Vorticity:** Uses `SymLogNorm` and `RdBu_r` so that both the weak interior core ($|\omega| \sim 1 - 5$) and ultra-high wall shear ($|\omega| \sim 200 - 1000$) are simultaneously visible without any washout.
3. **Centerline Profiles:** Centerline velocities $u(0.5, y)$ and $v(x, 0.5)$ compared with benchmark data.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

fig, axs = plt.subplots(1, 3, figsize=(22, 6.8), dpi=150)
X, Y, psi, omega, u, v = solver.X, solver.Y, solver.psi, solver.omega, solver.u, solver.v

# ----------------------------------------------------
# 1. STREAMLINES (Zero Blank Space, Clean Presentation)
# ----------------------------------------------------
ax1 = axs[0]
psi_min = float(psi.min())
psi_max = float(psi.max())

# High-density power-law levels to fill outer wall boundaries & core
n_neg = 55
neg_p = np.linspace(0.03, 1.0, n_neg)**2.2
levels_neg = np.sort(- neg_p * (-psi_min))

ax1.contourf(X, Y, psi, levels=levels_neg, cmap='viridis', alpha=0.9)
ax1.contour(X, Y, psi, levels=levels_neg, colors='black', linewidths=0.5, alpha=0.45)

# Multi-decade logarithmic levels for all secondary & tertiary eddies
if psi_max > 1e-9:
    levels_pos = np.logspace(-9, np.log10(max(psi_max, 1e-6)), 35)
    ax1.contourf(X, Y, psi, levels=levels_pos, cmap='autumn_r', alpha=0.95)
    ax1.contour(X, Y, psi, levels=levels_pos, colors='darkred', linewidths=0.6, alpha=0.6)

# Separatrix
ax1.contour(X, Y, psi, levels=[0.0], colors='white', linewidths=1.5, linestyles='--')

ax1.set_title(f"Streamlines ($\psi$) --- All Eddies Resolved\n$Re = {TARGET_RE:,}$ ($N = {TARGET_N} \\times {TARGET_N}$)", fontsize=12, fontweight='bold')
ax1.set_xlabel("$x / L$", fontsize=11)
ax1.set_ylabel("$y / L$", fontsize=11)
ax1.set_aspect('equal')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

# ----------------------------------------------------
# 2. VORTICITY (Symmetric Logarithmic Scale - No Washout)
# ----------------------------------------------------
ax2 = axs[1]
v_disp = 150.0
norm_v = SymLogNorm(linthresh=1.0, linscale=0.5, vmin=-v_disp, vmax=v_disp, base=10)
levels_v = np.unique(np.concatenate([
    -np.logspace(np.log10(v_disp), np.log10(0.05), 45),
    np.linspace(-0.05, 0.05, 11),
    np.logspace(np.log10(0.05), np.log10(v_disp), 45)
]))

im_v = ax2.contourf(X, Y, omega, levels=levels_v, norm=norm_v, cmap='RdBu_r', extend='both')
ax2.contour(X, Y, omega, levels=[-10, -5, -2, -1, 1, 2, 5, 10], colors='black', linewidths=0.35, alpha=0.3)
ax2.set_title(f"Vorticity ($\omega$) --- SymLog Scale\n(Core & Boundary Layer Shear)", fontsize=12, fontweight='bold')
ax2.set_xlabel("$x / L$", fontsize=11)
ax2.set_ylabel("$y / L$", fontsize=11)
ax2.set_aspect('equal')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
cb = plt.colorbar(im_v, ax=ax2, fraction=0.046, pad=0.04)
cb.set_label(r"Vorticity $\omega$ (SymLog Scale)")

# ----------------------------------------------------
# 3. CENTERLINE PROFILES
# ----------------------------------------------------
ax3 = axs[2]
mid_idx = TARGET_N // 2
ax3.plot(u[:, mid_idx], solver.y, 'b-', linewidth=2.2, label=r"$u(0.5, y)$ vertical cut")
ax3.plot(solver.x, v[mid_idx, :], 'r-', linewidth=2.2, label=r"$v(x, 0.5)$ horizontal cut")
ax3.set_title(f"Centerline Velocity Profiles\n$Re = {TARGET_RE:,}$ ($N = {TARGET_N}$)", fontsize=12, fontweight='bold')
ax3.set_xlabel("Velocity Magnitude", fontsize=11)
ax3.set_ylabel("Coordinate ($x$ or $y$)", fontsize=11)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.legend(loc='best', fontsize=10)

plt.tight_layout()
plt.show()


## 7. Automated Benchmark Verification vs. Erturk (2009, Table 3)
Evaluates vorticity along $y = 0.95$ across 26 stations and computes interior $L_2$ and MAE errors against Dr. Ercan Erturk's official benchmark data (*acenumerics.com*).


In [ ]:
from run_ultra_resolution_1025 import verify_against_erturk_table3

# Audit computed ultra-fine solution
audit_results = verify_against_erturk_table3(out_file)


## 8. Multi-Cycle Extended Unsteady Marching Pipeline ($Re = 100,000$, $50,000$, $25,000$ on $513 \times 513$)
Executes physical time-accurate Navier-Stokes marching over multi-cycle duration ($3 - 5$ full shedding periods per regime).
Reconstructs closed limit-cycle phase portraits, computes high-resolution FFT power spectrum, extracts 4-phase cycle decomposition, and renders synchronized publication animated GIFs for all high-$Re$ regimes.


In [ ]:
# Execute Multi-Cycle Extended Unsteady Marching Pipeline
# Captures 3 to 5 complete shedding cycles across extreme Reynolds numbers (Re = 100,000, 50,000, 25,000)
# Resolves closed limit-cycle attractors, multi-harmonic folds, 4-phase snapshots, and 48-frame GIFs!
!python scripts/run_colab_pipeline.py


## 9. Inline Display: Shedding Cycle Snapshots & Synchronized Animated GIF
Visualizes the dynamic 4-phase cycle snapshots and displays the publication-grade animated GIF with real-time probe telemetry tracking along $t/T \in [0, 1]$.


In [ ]:
import os
from IPython.display import Image, display

# Display multi-cycle phase portraits, cycle snapshots, and animated GIFs
for re_val in [100000, 50000, 25000]:
    print("=" * 70)
    print(f"DYNAMIC UNSTEADY ATTRACTOR RESULTS: Re = {re_val:,} (513 x 513)")
    print("=" * 70)
    for fig_file in [
        f"figures/lid_driven_unsteady_phase_portrait_Re{re_val}_N513.png",
        f"figures/lid_driven_unsteady_cycle_snapshots_Re{re_val}_N513.png",
        f"figures/lid_driven_unsteady_timeseries_Re{re_val}_N513.png",
        f"figures/lid_driven_vortex_shedding_Re{re_val}_N513.gif"
    ]:
        if os.path.exists(fig_file):
            print(f"Displaying: {fig_file}")
            display(Image(filename=fig_file))


## 10. Backup Solutions to Google Drive / Direct Export
Archive all newly computed flow fields and figures into a single compressed `.zip` archive for immediate download or copy to Google Drive.


In [ ]:
import os
try:
    from google.colab import files
    zip_name = 'colab_unsteady_results.zip'
    if os.path.exists(zip_name):
        print(f"Initiating download for '{zip_name}' ({os.path.getsize(zip_name)/(1024*1024):.2f} MB)...")
        files.download(zip_name)
    else:
        print(f"Archive '{zip_name}' not found. Check pipeline execution.")
except ImportError:
    print("Running locally; archive saved in root directory.")
